# Musicm8 — AI Producer + Inverse Sound Designer

This is the primary workflow:

**your reference songs → stems/MIDI/EnCodec links/spectral fingerprints → local pretrained producer AI → coherent song plan → editable MIDI → inverse sound matching → Musicm8 native synth + FX → audio stems + master**

The new **inverse sound designer** does what we discussed: for each selected reference role (drums, bass, chords, melody), Musicm8 takes a short aligned stem + MIDI window, renders its own synth, compares the result against the real reference using log-mel frequency shape, amplitude envelope, transients and loudness, changes synth/FX parameters, and repeats. The best patch is cached in Drive and reused later.

Change only `IDEA` to start. Increasing `MATCH_ITERS` searches harder but takes longer.

> Colab still has to allocate a GPU. If CUDA is unavailable choose **Runtime → Change runtime type → GPU**, then run the same cell again.


In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK AI PRODUCER + INVERSE SOUND MATCHING
# ============================================================

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

# ------------------------------------------------------------
# EDIT THIS
# ------------------------------------------------------------
IDEA = "dark UK garage track, emotional chords, deep moving bass, spacious pads"
BARS = 32
SEED = 42

# More iterations = more synth/FX guesses against each reference stem.
# 40 is a sensible first pass; 80-120 searches harder.
MATCH_ITERS = 40
MATCH_SECONDS = 3.0

AI_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO = ROOT / "audio"
WORK = ROOT / "work"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"

AUDIO.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

# Persist the pretrained producer brain and matched synth patches across resets.
HF_CACHE = WORK / "hf_cache"
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Always pull the latest Musicm8 code.
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)

print("\nInstalling Musicm8 dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ai.txt"],
    check=True,
)

import torch
print("\nPython:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU connected. Choose Runtime → Change runtime type → GPU, "
        "then run THIS SAME CELL again."
    )
print("GPU:", torch.cuda.get_device_name(0))

audio_files = [
    p for p in AUDIO.rglob("*")
    if p.is_file() and p.suffix.lower() in {".wav", ".mp3", ".flac", ".m4a", ".aac", ".ogg", ".opus"}
]
print("Reference songs:", len(audio_files))
if not audio_files:
    raise FileNotFoundError(f"Put your reference songs in {AUDIO}")

cmd = [
    sys.executable, "-u", "ai_producer_workflow.py",
    "--root", str(ROOT),
    "--repo", str(REPO),
    "--idea", IDEA,
    "--bars", str(BARS),
    "--seed", str(SEED),
    "--ai-model", AI_MODEL,
    "--match-iters", str(MATCH_ITERS),
    "--match-seconds", str(MATCH_SECONDS),
]

print("\n🚀 Starting Musicm8 AI producer...")
subprocess.run(cmd, check=True)

PROJECT = WORK / "ai_projects/latest"
MASTER = PROJECT / "master.wav"
MATCHED = PROJECT / "matched_patches.json"

print("\n✅ MUSICM8 AI PROJECT")
print("Idea        :", IDEA)
print("Plan        :", PROJECT / "plan.json")
print("References  :", WORK / "reference_library/library.json")
print("MIDI        :", PROJECT / "arrangement.mid")
print("MIDI stems  :", PROJECT / "midi_stems")
print("Matched     :", MATCHED)
print("A/B matches :", PROJECT / "sound_matches")
print("Audio stems :", PROJECT / "audio_stems")
print("Synth patch :", PROJECT / "synth_patches.json")
print("Master      :", MASTER)

from IPython.display import Audio, display

if MATCHED.exists():
    data = json.loads(MATCHED.read_text(encoding="utf-8"))
    print("\n🎛️ INVERSE SOUND-DESIGN RESULTS")
    for role, result in data.get("roles", {}).items():
        print(
            f"{role:7s} | distance {result.get('initial_distance')} → {result.get('best_distance')} "
            f"| search improvement {result.get('search_improvement_percent')}%"
        )

# Play the matched bass A/B first because it is usually easiest to judge.
for role in ("bass", "drums"):
    ref = PROJECT / "sound_matches" / f"{role}_reference.wav"
    matched = PROJECT / "sound_matches" / f"{role}_matched.wav"
    if ref.exists() and matched.exists():
        print(f"\n▶️ {role.upper()} REFERENCE EXCERPT")
        display(Audio(str(ref)))
        print(f"▶️ MUSICM8 MATCHED {role.upper()} PATCH")
        display(Audio(str(matched)))

if MASTER.exists():
    print("\n🎵 MUSICM8 FINAL MASTER")
    display(Audio(str(MASTER))
)


## Optional: inspect saved project files


In [ ]:
from pathlib import Path
root = Path("/content/drive/MyDrive/Musicm8/work")
project = root / "ai_projects/latest"
print("Reference library :", root / "reference_library/library.json")
print("Sound patch cache :", root / "sound_patch_cache")
print("Latest AI project :", project)
for p in sorted(project.rglob("*")) if project.exists() else []:
    if p.is_file():
        print(" -", p)
